In [104]:
# A gestão quer receber um resumo com os principais indicadores do período.

# A empresa precisa responder:

# Qual foi o faturamento total?

# Quantas ordens foram finalizadas?

# Qual foi o ticket médio geral?

# Qual foi o cliente com maior faturamento?

# Qual foi a cidade com maior faturamento?

# Qual foi a categoria com maior faturamento?

# Qual foi a primeira data de venda finalizada?

# Qual foi a última data de venda finalizada?

In [105]:
# lendo csv
import pandas as pd

clientes = pd.read_csv('../data/clientes.csv') 
ordens_servico = pd.read_csv('../data/ordens_servico.csv') #tabela fato
servicos = pd.read_csv('../data/servicos.csv') 


In [106]:
# unindo tabelas 
df = ordens_servico.merge(clientes, how='left', on='id_cliente')
df = df.merge(servicos, how='left', on='id_servico')


In [107]:
# filtrando status
df = df[df['status'] == 'Finalizado']


In [108]:
# ordenando tabela

df = df[['id_os', 'id_cliente', 'id_servico', 'nome_cliente', 'cidade', 'categoria', 'nome_servico', 'valor', 'data_os', 'status']]
display(df.head(3))

,id_os,id_cliente,id_servico,nome_cliente,cidade,categoria,nome_servico,valor,data_os,status
0,1,1,1,João,São Paulo,Manutenção,Troca de Óleo,120,2026-01-05,Finalizado
1,2,2,2,Maria,São Paulo,Serviços Rápidos,Alinhamento,80,2026-01-05,Finalizado
2,3,1,3,João,São Paulo,Segurança,Freio,350,2026-01-06,Finalizado


In [109]:
# Qual foi o faturamento total?

faturamento_total = df['valor'].sum()

display(faturamento_total)

np.int64(4620)

In [110]:
# Quantas ordens foram finalizadas?
ordens_finalizadas = df['id_os'].count()
display(ordens_finalizadas)

np.int64(15)

In [111]:
# Qual foi o ticket médio geral?
ticket_medio = df['valor'].mean()
display(ticket_medio)

np.float64(308.0)

In [112]:
# Qual foi o cliente com maior faturamento?
top_cliente = df.groupby('nome_cliente')['valor'].sum().idxmax()
display(top_cliente)

'João'

In [113]:
# Qual foi a cidade com maior faturamento?
top_cidade = df.groupby('cidade')['valor'].sum().idxmax()
display(top_cidade)

'São Paulo'

In [114]:
# Qual foi a categoria com maior faturamento?
top_categoria = df.groupby('categoria')['valor'].sum().idxmax()
display(top_categoria)

'Manutenção'

In [115]:
# Qual foi a primeira data de venda finalizada?

#convertendo str para data
df['data_os'] = pd.to_datetime(df['data_os'])

# analisando data
primeira_data = df.groupby('data_os')['data_os'].first().idxmin()

display(primeira_data)

Timestamp('2026-01-05 00:00:00')

In [116]:
# Qual foi a última data de venda finalizada?
ultima_data = df.groupby('data_os')['data_os'].first().idxmax()
display(ultima_data)

Timestamp('2026-01-22 00:00:00')

In [117]:
# relatorio_clientes
relatorio_clientes = (df.groupby('nome_cliente').agg(
    id_cliente=('id_cliente', 'first'),
    cidade=('cidade', 'first'),
    faturamento_total=('valor', 'sum'),
    qtd_ordens=('id_os', 'count'),
    ticket_medio=('valor', 'mean')
).reset_index().sort_values('faturamento_total', ascending=False))

display(relatorio_clientes)


,nome_cliente,id_cliente,cidade,faturamento_total,qtd_ordens,ticket_medio
2,João,1,São Paulo,1440,4,360.0
3,Juliana,6,Guarulhos,1050,3,350.0
4,Maria,2,São Paulo,1010,4,252.5
0,Ana,3,Guarulhos,590,2,295.0
5,Pedro,5,Mogi das Cruzes,450,1,450.0
1,Carlos,4,Suzano,80,1,80.0


In [118]:
# relatorio_categorias
relatorio_categorias = df.groupby('categoria').agg(
    faturamento_total=('valor', 'sum'),
    qtd_ordens=('id_os', 'count'),
    ticket_medio=('valor', lambda x: x.mean().round(2))
)
relatorio_categorias['participacao_percentual'] = (((df.groupby('categoria')['valor'].sum())/faturamento_total)*100).round(2)
relatorio_categorias = relatorio_categorias.reset_index().sort_values('faturamento_total', ascending=False)
display(relatorio_categorias)

,categoria,faturamento_total,qtd_ordens,ticket_medio,participacao_percentual
1,Manutenção,2420,7,345.71,52.38
2,Segurança,1030,3,343.33,22.29
0,Elétrica,920,2,460.00,19.91
3,Serviços Rápidos,250,3,83.33,5.41


In [119]:
# relatorio_cidades
relatorio_cidades = df.groupby('cidade').agg(
    faturamento_total=('valor', 'sum'),
    qtd_ordens=('id_os', 'count'),
    ticket_medio=('valor', lambda x: x.mean().round(2))
)

relatorio_cidades['participacao_percentual'] = (((df.groupby('cidade')['valor'].sum())/faturamento_total)*100).round(2)
relatorio_cidades = relatorio_cidades.reset_index().sort_values('faturamento_total', ascending=False)

display(relatorio_cidades)

,cidade,faturamento_total,qtd_ordens,ticket_medio,participacao_percentual
3,São Paulo,2450,8,306.25,53.03
0,Guarulhos,1640,5,328.00,35.50
1,Mogi das Cruzes,450,1,450.00,9.74
2,Suzano,80,1,80.00,1.73


In [120]:
# resumo_executivo
resumo_executivo = pd.DataFrame({
    'metrica': [
    'faturamento_total',
    'qtd_ordens_finalizadas',
    'ticket_medio_geral',
    'cliente_maior_faturamento',
    'cidade_maior_faturamento',
    'categoria_maior_faturamento',
    'primeira_data_finalizada',
    'ultima_data_finalizada'
], 'valor': [
    faturamento_total,
    ordens_finalizadas,
    ticket_medio,
    top_cliente,
    top_cidade,
    top_categoria,
    primeira_data,
    ultima_data
]
})



In [121]:
# exportando relatorios

resumo_executivo.to_csv('../output/resumo_executivo.csv', index=False, date_format='%d/%m/%Y')

relatorio_clientes.to_csv('../output/relatorio_clientes.csv', index=False, date_format='%d/%m/%Y')

relatorio_categorias.to_csv('../output/relatorio_categorias.csv', index=False, date_format='%d/%m/%Y')

relatorio_cidades.to_csv('../output/relatorio_cidades.csv', index=False, date_format='%d/%m/%Y')